In [3]:
!pip install -q "datasets<3.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [4]:
!pip install razdel

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [2]:
from datasets import load_dataset
from razdel import tokenize
import pandas as pd
import numpy as np


def token_len(text):
    return len(list(tokenize(text)))


def calc_stats(dataset_split):

    text_lens = []
    sum_lens = []

    for item in dataset_split:

        text_lens.append(
            token_len(item["text"])
        )

        sum_lens.append(
            token_len(item["summary"])
        )

    return {
        "Text mean": round(np.mean(text_lens), 2),
        "Text min": int(np.min(text_lens)),
        "Text max": int(np.max(text_lens)),
        "Summary mean": round(np.mean(sum_lens), 2),
        "Summary min": int(np.min(sum_lens)),
        "Summary max": int(np.max(sum_lens))
    }


datasets_info = {
    "Gazeta":
        load_dataset(
            "IlyaGusev/gazeta",
            revision="v2.0"
        ),

    "MLSUM":
        load_dataset(
            "reciTAL/mlsum",
            "ru"
        ),

    "XLSUM":
        load_dataset(
            "csebuetnlp/xlsum",
            "russian"
        )
}




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


The repository for IlyaGusev/gazeta contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/IlyaGusev/gazeta.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/757 [00:00<?, ? examples/s]

The repository for csebuetnlp/xlsum contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/csebuetnlp/xlsum.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [3]:
rows = []

for dataset_name, ds in datasets_info.items():
    print(f"Dataset: {dataset_name}")
    for split in ["train", "validation", "test"]:
        print(f"Split: {split}")
        stats = calc_stats(ds[split])

        rows.append({
            "Dataset": dataset_name,
            "Split": split,
            **stats
        })



Dataset: Gazeta
Split: train
Split: validation
Split: test
Dataset: MLSUM
Split: train
Split: validation
Split: test
Dataset: XLSUM
Split: train
Split: validation
Split: test


In [4]:
df = pd.DataFrame(rows)

display(df)

df.to_excel(
    "dataset_statistics.xlsx",
    index=False
)

,Dataset,Split,Text mean,Text min,Text max,Summary mean,Summary min,Summary max
0,Gazeta,train,766.88,28,1500,49.57,15,85
1,Gazeta,validation,723.95,344,1500,53.09,15,85
2,Gazeta,test,732.04,246,1500,53.91,15,85
3,MLSUM,train,949.86,55,11689,14.66,10,65
4,MLSUM,validation,1156.67,118,5842,13.43,10,30
5,MLSUM,test,1214.40,69,26794,13.44,10,35
6,XLSUM,train,682.14,19,22274,29.40,1,246
7,XLSUM,validation,556.87,62,1583,27.94,8,60
8,XLSUM,test,555.84,54,1745,27.93,8,60


In [6]:
import os
print("Загрузка IIS датасета")
os.system("rm -rf /content/summarization-dataset")
os.system("git clone https://github.com/iis-research-team/summarization-dataset.git /content/summarization-dataset")

texts_iis, abstracts_iis = [], []
data_path = "/content/summarization-dataset/dataset"

for domain in os.listdir(data_path):
    domain_path = os.path.join(data_path, domain)
    if os.path.isdir(domain_path):
        for paper_folder in os.listdir(domain_path):
            paper_path = os.path.join(domain_path, paper_folder)
            text_file = os.path.join(paper_path, "text.txt")
            abstract_file = os.path.join(paper_path, "abstract.txt")

            if os.path.exists(text_file) and os.path.exists(abstract_file):
                with open(text_file, "r", encoding="utf-8") as f:
                    text = f.read().strip()
                with open(abstract_file, "r", encoding="utf-8") as f:
                    abstract = f.read().strip()

                if text and abstract:
                    texts_iis.append(text)
                    abstracts_iis.append(abstract)

print(f"IIS: загружено {len(texts_iis)} пар")

Загрузка IIS датасета
IIS: загружено 479 пар


In [7]:
from razdel import tokenize
import numpy as np

def token_len(text):
    return len(list(tokenize(text)))

iis_text_lengths = [token_len(t) for t in texts_iis]
iis_summary_lengths = [token_len(s) for s in abstracts_iis]

iis_stats = {
    "Dataset": "IIS",
    "Split": "all",
    "Samples": len(texts_iis),

    "Text mean": round(np.mean(iis_text_lengths), 2),
    "Text min": int(np.min(iis_text_lengths)),
    "Text max": int(np.max(iis_text_lengths)),

    "Summary mean": round(np.mean(iis_summary_lengths), 2),
    "Summary min": int(np.min(iis_summary_lengths)),
    "Summary max": int(np.max(iis_summary_lengths))
}

print(iis_stats)

{'Dataset': 'IIS', 'Split': 'all', 'Samples': 479, 'Text mean': np.float64(3219.11), 'Text min': 725, 'Text max': 8517, 'Summary mean': np.float64(115.5), 'Summary min': 26, 'Summary max': 557}


In [8]:
rows.append(iis_stats)

In [9]:
df = pd.DataFrame(rows)

display(df)

df.to_excel(
    "dataset_statistics.xlsx",
    index=False
)

,Dataset,Split,Text mean,Text min,Text max,Summary mean,Summary min,Summary max,Samples
0,Gazeta,train,766.88,28,1500,49.57,15,85,NaN
1,Gazeta,validation,723.95,344,1500,53.09,15,85,NaN
2,Gazeta,test,732.04,246,1500,53.91,15,85,NaN
3,MLSUM,train,949.86,55,11689,14.66,10,65,NaN
4,MLSUM,validation,1156.67,118,5842,13.43,10,30,NaN
5,MLSUM,test,1214.40,69,26794,13.44,10,35,NaN
6,XLSUM,train,682.14,19,22274,29.40,1,246,NaN
7,XLSUM,validation,556.87,62,1583,27.94,8,60,NaN
8,XLSUM,test,555.84,54,1745,27.93,8,60,NaN
9,IIS,all,3219.11,725,8517,115.50,26,557,479.0


In [10]:
from datasets import Dataset, DatasetDict

iis_dataset = Dataset.from_dict({
    "text": texts_iis,
    "summary": abstracts_iis
})

# 80 / 10 / 10

tmp = iis_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

test_valid = tmp["test"].train_test_split(
    test_size=0.5,
    seed=42
)

iis_dataset = DatasetDict({
    "train": tmp["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

print(iis_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 383
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 48
    })
    test: Dataset({
        features: ['text', 'summary'],
        num_rows: 48
    })
})


In [11]:
from razdel import tokenize
import numpy as np

def token_len(text):
    return len(list(tokenize(text)))

rows = []

for split_name in ["train", "validation", "test"]:

    text_lens = [
        token_len(x["text"])
        for x in iis_dataset[split_name]
    ]

    summary_lens = [
        token_len(x["summary"])
        for x in iis_dataset[split_name]
    ]

    rows.append({
        "Dataset": "IIS",
        "Split": split_name,
        "Samples": len(iis_dataset[split_name]),

        "Text mean": round(np.mean(text_lens), 2),
        "Text min": int(np.min(text_lens)),
        "Text max": int(np.max(text_lens)),

        "Summary mean": round(np.mean(summary_lens), 2),
        "Summary min": int(np.min(summary_lens)),
        "Summary max": int(np.max(summary_lens))
    })

In [12]:
df = pd.DataFrame(rows)

display(df)

df.to_excel(
    "dataset_statistics.xlsx",
    index=False
)

,Dataset,Split,Samples,Text mean,Text min,Text max,Summary mean,Summary min,Summary max
0,IIS,train,383,3222.30,739,8517,115.34,26,557
1,IIS,validation,48,3278.21,1261,6185,106.25,49,326
2,IIS,test,48,3134.58,725,7803,125.96,26,449
